# 2. 流程增强

这一节解决的问题是：**一次检索 + 一次生成不够时，系统应该如何多走几步。**

## 与第4章的边界

- 第4章强调：如何把 `query` 变得更适合检索。
- 本节强调：系统拿到中间结果后，如何继续决策下一步流程。

In [ ]:
from pathlib import Path

from dotenv import load_dotenv
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import PyPDFLoader
from langchain_community.vectorstores import Chroma

load_dotenv()

MODEL_NAME = "gpt-4o-mini"
PDF_PATH = Path("notebook/C7 高级 RAG 技巧/6. 增强阶段/data/face.pdf")

llm = ChatOpenAI(model=MODEL_NAME, temperature=0)
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

docs = PyPDFLoader(str(PDF_PATH)).load()
splitter = RecursiveCharacterTextSplitter(chunk_size=220, chunk_overlap=40)
chunks = splitter.split_documents(docs)
vs = Chroma.from_documents(chunks, embedding=embeddings)
retriever = vs.as_retriever(search_kwargs={"k": 4})

## 单轮流程失败示例（Baseline）

问题：`请先归纳文档的核心矛盾，再给出按优先级排序的改进路径。`

这类问题通常至少需要：
1) 先抽取核心矛盾；
2) 再针对每个矛盾补检索证据；
3) 最后综合排序。

单轮“检索一次 + 生成一次”经常只覆盖其中一部分。

In [ ]:
baseline_query = "请先归纳文档的核心矛盾，再给出按优先级排序的改进路径。"
baseline_docs = retriever.invoke(baseline_query)
baseline_context = "\n\n".join(d.page_content for d in baseline_docs)

baseline_prompt = f"""
仅根据上下文回答：
问题：{baseline_query}
上下文：
{baseline_context}
"""

baseline_answer = llm.invoke(baseline_prompt).content
print("=== baseline answer ===")
print(baseline_answer)

### Baseline 失败分析

观察输出你会发现，baseline 的回答往往只抓住了问题的某一个维度——比如列出了核心矛盾，但缺少对应的证据支撑；或者给了改进方向，但没有按优先级排序。这不是上下文不全的问题（chunk 大小 220 已经足够包含完整段落），而是**流程本身只走了一步**：一次检索只能覆盖一个维度的信息。

要想让系统在面对这类多维度问题时给出完整答案，我们需要让它"多走几步"——这就是流程增强要解决的核心问题。下面的六种方法各自用不同的策略来实现这一目标。

## 五种方法的流程控制分类

| 方法 | 控制类型 | 关键决策点 |
|---|---|---|
| 迭代检索 | 补检索型 | "还缺什么？" -> 继续/停止 |
| 递归检索 | 拆任务型 | "该拆成哪些子问题？" |
| 查询路由 | 选路径型 | "走哪条检索链路？" |
| Corrective RAG | 控质量型 | "检索结果够好吗？" -> 过滤/补检 |
| Self-RAG | 自反思型 | "回答够好吗？" -> 继续检索/停止 |
| 自适应检索 | 选深度型 | "这个问题有多复杂？" -> 选择检索策略 |


## 迭代检索（Iterative Retrieval）

### 失败场景
单轮回答只覆盖了问题的一部分，遗漏关键证据。

### 循环流程
1. 初次检索并生成草答
2. 让 LLM 指出“还缺什么”
3. 用缺失点继续检索
4. 合并上下文再回答

### 边界
- 适合：信息可逐步补齐的问题
- 局限：轮次越多，延迟与成本越高

### 关键决策点
当前答案是否仍有关键缺口，若有则继续检索，否则停止。


In [ ]:
def iterative_retrieval(query: str, max_rounds: int = 2):
    context_blocks = []
    missing_hint = ""

    for _ in range(max_rounds):
        effective_query = query if not missing_hint else f"{query}\n补充检索线索：{missing_hint}"
        new_docs = retriever.invoke(effective_query)
        context_blocks.extend(d.page_content for d in new_docs)
        merged_context = "\n\n".join(context_blocks[-8:])

        draft_prompt = f"""
问题：{query}
上下文：
{merged_context}
请先给出当前答案，再用一句话指出仍缺失的关键信息。
输出格式：
当前答案：...
缺失信息：...
"""
        draft = llm.invoke(draft_prompt).content

        if "缺失信息：无" in draft or "缺失信息: 无" in draft:
            return draft

        missing_hint = draft.split("缺失信息")[-1].strip("：: \n")

    final_prompt = f"问题：{query}\n上下文：\n{merged_context}\n请给出最终完整回答。"
    return llm.invoke(final_prompt).content

iterative_answer = iterative_retrieval(baseline_query, max_rounds=2)
print("=== baseline ===")
print(baseline_answer[:220])
print("\n=== iterative ===")
print(iterative_answer[:320])

### 迭代检索结果分析

对比 baseline 和迭代检索的输出可以看到，迭代检索通过"先答后补"逐步完善了答案的覆盖度。第一轮给出初步回答并指出缺失，第二轮根据缺失线索补充检索，最终合并的上下文比单轮检索更全面。

不过迭代检索的子问题是**隐式产生**的——由 LLM 自行判断"还缺什么"。如果问题本身就可以显式拆解为独立的子问题（"核心矛盾是什么"、"每个矛盾的证据"、"改进优先级"），直接拆比让模型自己猜更可控。这就是递归检索的思路。

## 递归检索（Recursive Retrieval）

### 失败场景
复杂问题包含多个子目标，单次检索无法覆盖全部维度。

### 分解流程
1. 主问题拆成子问题
2. 每个子问题独立检索与回答
3. 聚合子答案得到最终回答

### 子问题示例
- 核心问题是什么？
- 对应证据有哪些？
- 先做哪些改进最稳妥？

### 边界
- 适合：可显式分解的问题
- 局限：子问题质量决定上限

### 关键决策点
主问题是否可有效拆解为可检索、可验证的子问题。


In [ ]:
def recursive_retrieval(main_question: str, sub_questions: list[str]):
    sub_answers = []
    for sq in sub_questions:
        sq_docs = retriever.invoke(sq)
        sq_ctx = "\n\n".join(d.page_content for d in sq_docs)
        sq_prompt = f"子问题：{sq}\n上下文：\n{sq_ctx}\n请仅回答该子问题。"
        sub_answers.append({"sub_question": sq, "answer": llm.invoke(sq_prompt).content})

    merge_prompt = f"""
主问题：{main_question}
子答案：{sub_answers}
请整合为一个结构化回答（问题 -> 证据 -> 建议顺序）。
"""
    return sub_answers, llm.invoke(merge_prompt).content

sub_questions = [
    "文档中指出的核心矛盾是什么？",
    "每个矛盾对应的关键证据有哪些？",
    "改进路径应该如何排序？",
]
sub_answers, recursive_answer = recursive_retrieval(baseline_query, sub_questions)
print("子问题数量:", len(sub_answers))
print("=== recursive answer ===")
print(recursive_answer[:320])

### 递归检索结果分析

递归检索的优势在于每个子问题都有独立的检索上下文，不会互相干扰。最终合成时 LLM 可以看到所有子答案，给出更结构化的回答。

但迭代和递归检索都在解决"多走几步"的问题，还没回答一个更基础的问题：**该走哪条路？** 当系统有多个索引或多个工具时，选错检索链路会导致无论走多少步都找不到对的信息。这就是查询路由要解决的问题。

## 查询路由与自适应检索（Query Routing & Adaptive Retrieval）

查询路由解决"走哪条检索链路"，自适应检索解决"走多深"。两者都是检索前的策略决策，放在一起形成完整的"检索前决策层"。

### 规则路由
按关键词把问题分发到不同索引/工具，简单稳定。

### LLM 路由
先让 LLM 判断“该走哪条检索链路”，再执行对应分支，灵活但更依赖模型判断。

### 边界
- 适合：多知识源、多工具场景
- 局限：错误路由会直接影响最终答案

### 关键决策点
当前问题应走哪条索引/工具链路，才能用最少噪声拿到证据。


In [ ]:
doc_text = "\n".join(d.page_content for d in docs)
parts = [p.strip() for p in doc_text.split("\n\n") if p.strip()]
method_texts = [p for p in parts if "算法" in p or "模型" in p][:80] or parts[:80]
concept_texts = [p for p in parts if "区别" in p or "定义" in p][:80] or parts[:80]

method_vs = Chroma.from_texts(method_texts, embedding=embeddings)
concept_vs = Chroma.from_texts(concept_texts, embedding=embeddings)

method_retriever = method_vs.as_retriever(search_kwargs={"k": 3})
concept_retriever = concept_vs.as_retriever(search_kwargs={"k": 3})

def rule_route(query: str) -> str:
    if any(k in query for k in ["步骤", "流程", "算法"]):
        return "method"
    return "concept"

def llm_route(query: str) -> str:
    judge_prompt = f"""
你是路由器，只输出 method 或 concept。
query: {query}
如果问题更关注步骤/流程输出 method，否则输出 concept。
"""
    decision = llm.invoke(judge_prompt).content.strip().lower()
    return "method" if "method" in decision else "concept"

def routed_answer(query: str, use_llm_router: bool = True) -> str:
    route = llm_route(query) if use_llm_router else rule_route(query)
    r = method_retriever if route == "method" else concept_retriever
    route_docs = r.invoke(query)
    ctx = "\n\n".join(d.page_content for d in route_docs)
    return route, llm.invoke(f"问题：{query}\n上下文：\n{ctx}\n请回答。").content

route_query = "请解释 transformer 的核心机制与执行流程。"
route_name, routed_resp = routed_answer(route_query, use_llm_router=True)
print('route:', route_name)
print(routed_resp[:280])

### 查询路由结果分析

查询路由的价值在于：当系统有多个专用索引时，先选对索引再检索，比把所有内容混在一起检索更精准。规则路由简单稳定但不够灵活，LLM 路由更通用但引入了模型判断的不确定性。

查询路由决定了"走哪条路"，但还有一个问题没解决：**走多深？** 一个简单的事实查询只需要一次检索就够了，但一个复杂的多维度问题可能需要迭代或递归检索。让系统根据问题复杂度自动选择检索深度，就是自适应检索要做的事情。

## 自适应检索（Adaptive Retrieval）

### 失败场景
对所有问题都用同一个检索深度——简单问题浪费资源，复杂问题覆盖不足。

### 思路
先让 LLM 判断问题的复杂度（simple / moderate / complex），再根据复杂度自动选择检索策略：简单问题单次检索，中等问题迭代检索，复杂问题递归检索。

### 与查询路由的关系
查询路由选择"走哪条路"（哪个索引/工具），自适应检索选择"走多深"（单次/迭代/递归）。两者组合形成完整的"检索前决策层"。

### 边界
- 适合：问题复杂度差异大的场景
- 局限：复杂度判断依赖 LLM，可能误判

### 关键决策点
当前问题需要多深的检索策略才能覆盖答案所需的全部证据。

In [ ]:
def classify_query_complexity(query: str) -> str:
    prompt = f"""
判断以下问题的复杂度，只输出 simple / moderate / complex 之一。
- simple: 单一事实查询，一次检索即可回答
- moderate: 需要多个证据片段，可能需要补充检索
- complex: 需要多步推理、子问题拆解或跨文档整合

问题：{query}
"""
    return llm.invoke(prompt).content.strip().lower()

def adaptive_retrieval_answer(query: str) -> str:
    complexity = classify_query_complexity(query)

    if complexity == "simple":
        docs_ = retriever.invoke(query)
        ctx = "\n\n".join(d.page_content for d in docs_)
        answer = llm.invoke(f"问题：{query}\n上下文：\n{ctx}\n请回答。").content
    elif complexity == "moderate":
        answer = iterative_retrieval(query, max_rounds=2)
    else:
        sub_qs = [
            f"关于'{query}'的核心概念是什么？",
            f"关于'{query}'的关键证据有哪些？",
            f"关于'{query}'的结论或建议是什么？",
        ]
        _, answer = recursive_retrieval(query, sub_qs)

    return complexity, answer

test_queries = [
    "什么是SVM？",
    "请归纳文档的核心矛盾，再给出改进路径。",
]
for tq in test_queries:
    comp, ans = adaptive_retrieval_answer(tq)
    print(f"query: {tq}")
    print(f"complexity: {comp}")
    print(f"answer: {ans[:200]}\n")

### 自适应检索结果分析

可以看到，简单问题被正确判定为 simple 并用单次检索高效处理，复杂问题则自动升级为递归检索以覆盖更多维度。自适应检索本质上是一个"调度器"，它复用了前面介绍的迭代和递归检索作为底层策略。

到目前为止，我们解决了"多走几步"和"走对方向"的问题。但还有一个更根本的问题没有触及：**检索回来的结果质量够不够好？** 如果检索结果里混入了大量低相关片段，无论后续流程多精巧，最终答案都可能被噪声带偏。Corrective RAG 就是在生成前加一道质量把关。

## Corrective RAG

### 失败场景
检索结果里混入低相关片段，导致答案偏题或不稳定。

### CRAG 流程
1. 先检索候选文档
2. 用 grader 评估每条证据相关性
3. 按 `全部相关 / 部分相关 / 无关` 分支处理
4. 再生成最终回答

### 直接收益
在不大改系统结构的前提下，先把“坏上下文”挡在生成前。

### 关键决策点
候选检索结果是全部可用、部分可用还是基本不可用。


In [ ]:
def grade_relevance(query: str, text: str) -> str:
    judge_prompt = f"""
判断下面文档片段和问题的相关性，只输出：relevant / partial / irrelevant
问题：{query}
片段：{text[:600]}
"""
    result = llm.invoke(judge_prompt).content.lower()
    if "irrelevant" in result:
        return "irrelevant"
    if "partial" in result:
        return "partial"
    return "relevant"

def corrective_rag_answer(query: str):
    docs_ = retriever.invoke(query)
    bucket = {"relevant": [], "partial": [], "irrelevant": []}
    for d in docs_:
        tag = grade_relevance(query, d.page_content)
        bucket[tag].append(d.page_content)

    if len(bucket["relevant"]) == len(docs_):
        context = "\n\n".join(bucket["relevant"])
        branch = "all_relevant"
    elif bucket["relevant"] or bucket["partial"]:
        context = "\n\n".join((bucket["relevant"] + bucket["partial"])[:4])
        branch = "partially_relevant"
    else:
        context = "检索证据不足，请先重写问题后再检索。"
        branch = "irrelevant"

    answer = llm.invoke(f"问题：{query}\n上下文：\n{context}\n请回答。") .content
    return branch, answer

crag_branch, crag_answer = corrective_rag_answer(baseline_query)
print('CRAG branch:', crag_branch)
print('\n=== baseline ===')
print(baseline_answer[:200])
print('\n=== CRAG ===')
print(crag_answer[:280])

## Self-RAG（教学化简版）

Self-RAG 的核心是三段循环：**retrieve -> generate -> critique**。

- 训练阶段：通过特定监督信号学习何时继续检索、何时停止。
- 推理阶段：根据当前回答质量决定是否追加检索。

下图用于理解原始论文思路（保留原图链接）：
- `./figures/selfrag.png`
- `./figures/selftoken.jpg`

本节不再要求本地 LLaMA2 / GGUF / 重型 pack 环境，只保留可运行控制逻辑示例。

### 关键决策点
当前回答质量是否足够，是否值得继续追加检索。


### Corrective RAG 结果分析

CRAG 的三种分支各有含义：
- **all_relevant**：所有检索结果都相关，直接用于生成——这是最理想的情况。
- **partially_relevant**：部分相关、部分噪声，过滤掉低质量片段后再生成——这是最常见的情况。
- **irrelevant**：检索结果基本不可用，应提示用户重写问题或触发补充检索。

CRAG 的决策是单次的：评估一次、处理一次。但如果我们希望系统能**持续自我判断**——"这个回答够好吗？要不要再检索一轮？"——就需要一个更动态的反思机制。这就是 Self-RAG 的核心思想。

In [ ]:
def self_rag_loop(query: str, max_rounds: int = 3):
    context_blocks = []
    answer = ""

    for _ in range(max_rounds):
        docs_ = retriever.invoke(query if not answer else f"{query}\n已有回答：{answer}")
        context_blocks.extend(d.page_content for d in docs_)
        context = "\n\n".join(context_blocks[-8:])

        answer = llm.invoke(f"问题：{query}\n上下文：\n{context}\n给出当前回答。") .content
        critique = llm.invoke(
            f"问题：{query}\n回答：{answer}\n是否还需继续检索？只输出 yes 或 no。"
        ).content.lower()

        if "no" in critique:
            break

    return answer

self_rag_answer = self_rag_loop(baseline_query)
print(self_rag_answer[:320])

## 小结：五种流程增强如何选择

- `迭代检索`：适合“先答后补”的渐进式问题。
- `递归检索`：适合可拆解的复杂问题。
- `查询路由`：适合多索引/多工具环境。
- `Corrective RAG`：适合先提升检索质量再生成。
- `Self-RAG`：适合需要模型自我反思控制检索轮次的场景。
- `自适应检索`：适合需要根据问题复杂度自动选择检索深度的场景，与查询路由配合使用。

## 学习检查点

- 你能描述 CRAG 的三种分支（all / partial / irrelevant）吗？
- 你能解释 Self-RAG 在“继续检索/停止检索”上的决策点吗？
- 你能解释查询路由（选方向）和自适应检索（选深度）的区别吗？
- 你知道何时应从流程增强升级到系统增强吗？

## 与系统增强的衔接

当你发现“仅靠流程决策还不够”，例如需要跨多个文档智能体协作、需要长期会话记忆时，请继续学习：`3. 系统增强.ipynb`。

### Self-RAG 结果分析

Self-RAG 的优势在于：模型自己决定何时停止检索，避免了固定轮次带来的浪费或不足。

**教学简化版 vs 论文原版的差异：** 本节用通用 LLM API 模拟了 Self-RAG 的 retrieve → generate → critique 循环。论文原版通过特殊的 reflection tokens（如 `[Retrieve]`、`[IsRel]`、`[IsSup]`）在模型内部完成这些决策，需要专门微调的模型（如 selfrag_llama2_7b）。教学版保留了核心控制逻辑，但决策精度依赖通用 LLM 的判断能力，而非内置的反思 token。

## 全方法对比表

| 方法 | 控制类型 | 典型修复问题 | 新增复杂度 | 最适合场景 |
|---|---|---|---|---|
| 迭代检索 | 补检索型 | 首轮回答不完整 | 中 | 信息可逐步补齐 |
| 递归检索 | 拆任务型 | 复杂问题可分解 | 中-高 | 多跳问答 |
| 查询路由 | 选路径型 | 多数据源或多索引 | 中 | 中大型系统 |
| 自适应检索 | 选深度型 | 问题复杂度差异大 | 中 | 与查询路由配合 |
| Corrective RAG | 控质量型 | 检索结果噪声多 | 中-高 | 质量优先场景 |
| Self-RAG | 自反思型 | 需要动态控制检索 | 高 | 高要求场景 |